# 6.1 Common Structured Query Language (SQL) statements

## Data Definition Language

In [ ]:
# Creates a table named Compounds
CREATE TABLE Compounds (
    CompoundID INT PRIMARY KEY,
    Name VARCHAR(100),
    MolecularFormula VARCHAR(50),
    MolecularWeight DECIMAL(10, 4),
    SMILES VARCHAR(255),
    InChI VARCHAR(255)
);

# Delete the table named Compounds
DROP TABLE Compounds;

# Add a new column LogP to the Compounds table
ALTER TABLE Compounds  
ADD LogP DECIMAL(6, 3);

## Data Manipulation Language

In [ ]:
# Insert new data into the Compounds table
INSERT INTO Compounds (CompoundID, Name, MolecularFormula, MolecularWeight, SMILES)  
VALUES (1, 'Water', 'H2O', 18.015, 'O');

# Update the SMILES column for a compound with CompoundID = 1
UPDATE Compounds  
SET SMILES = 'H[O]H'  
WHERE CompoundID = 1;

# Delete a compound with CompoundID = 101
DELETE FROM Compounds
WHERE CompoundID = 101;

## Data Control Language

In [ ]:
# Allow a user to perform specific operations on a database object
GRANT SELECT, INSERT ON Compounds TO 'user1';

# Remove the permissions
REVOKE INSERT ON Compounds FROM 'user1';

## Data Query Language

In [ ]:
# Retrieve the name of a compound with CompoundID = 1
SELECT Name, MolecularFormula  
FROM Compounds  
WHERE CompoundID = 101;

## Transaction Control Language

In [ ]:
# Set a point within a transaction that can be rolled back later
SAVEPOINT savepoint_name;

# roll back to a savepoint
ROLLBACK TO savepoint_name;

# 6.2 ORM example using SQLAlchemy in Python

In [4]:
from sqlalchemy import Column, Integer, String,Float, create_engine
from sqlalchemy.orm import sessionmaker, declarative_base

#Create the base class of the object
Base = declarative_base()

#Define the Compound object
class Compound(Base):
    __tablename__ = 'compound'
    id = Column(Integer, primary_key=True, autoincrement=True) 
    name = Column(String(100), nullable=False)
    molecular_formula = Column(String(50)) 
    molecular_weight = Column(Float) 
    smiles = Column(String(255)) 

engine = create_engine('sqlite:///chemistry.db')
Base.metadata.create_all(engine)
# Create a session to interact with the database
Session = sessionmaker(bind=engine)
session = Session()

# Insert a new compound
new_compound = Compound(
    name="Aspirin",
    molecular_formula="C9H8O4",
    molecular_weight=180.16,
    smiles="CC(=O)OC1=CC=CC=C1C(=O)O"
)
session.add(new_compound)
session.commit()


# 6.3 Compound Database Management and Vector Similarity Search Using SQLite and SQLAlchemy

In [3]:
from sqlalchemy import create_engine, Column, Integer, String, Float,LargeBinary, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker, relationship
import sqlite3
import numpy as np
import sqlite_vec
import struct

def serialize_f32(vector):
    """Serialize the float list to a BLOB"""
    return struct.pack("%sf" % len(vector), *vector)

# Create the base class of the object
Base = declarative_base()

# Define the Compound object
class Compound(Base):
    __tablename__ = 'compound'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    name = Column(String, nullable=False)
    smiles = Column(String, unique=True, nullable=False)
    inchi = Column(String, unique=True, nullable=False)

# Define the Property object
class Property(Base):
    __tablename__ = 'property'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    compound_id = Column(Integer, ForeignKey('compound.id'), nullable=False)
    property_type = Column(String, nullable=False)
    value = Column(Float, nullable=False)
    unit = Column(String, nullable=True)
    
    compound = relationship("Compound", back_populates="properties")

Compound.properties = relationship("Property", order_by=Property.id, back_populates="compound")

# Define the Spectrum object
class Spectrum(Base):
    __tablename__ = 'spectrum'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    compound_id = Column(Integer, ForeignKey('compound.id'), nullable=False)
    instrument = Column(String, nullable=False)
    collision_energy = Column(Float, nullable=True)
    binned_spectrum = Column(LargeBinary, nullable=False)
    
    compound = relationship("Compound", back_populates="spectra")

Compound.spectra = relationship("Spectrum", order_by=Spectrum.id, back_populates="compound")

# Define the Embedding object
class Embedding(Base):
    __tablename__ = 'embedding'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    compound_id = Column(Integer, ForeignKey('compound.id'), nullable=False)
    vector = Column(LargeBinary, nullable=False)  # Store as strings

    compound = relationship("Compound", back_populates="embedding")

Compound.embedding = relationship("Embedding", uselist=False, back_populates="compound")

# Create a SQLite database
engine = create_engine("sqlite:///chemistry2.db", echo=True)
Base.metadata.create_all(engine)

# Create a session to interact with the database
Session = sessionmaker(bind=engine)
session = Session()

# Insert sample data
compound = Compound(name="Caffeine", smiles="CN1C=NC2=C1C(=O)N(C(=O)N2C)C", inchi="InChI=1S/C8H10N4O2/c1-10-4-9-6-5(10)7(13)12(3)8(14)11(6)2/h4H,1-3H3")
session.add(compound)
session.commit()

# Insert property data
prop = Property(compound_id=compound.id, property_type="Molecular Weight", value=194.19, unit="g/mol")
session.add(prop)
session.commit()

# Insert mass spectrometry data
spec = Spectrum(compound_id=compound.id, instrument="Orbitrap", collision_energy=20.0, binned_spectrum=np.random.rand(100).astype(np.float32).tobytes())
session.add(spec)
session.commit()

# Compute and store the embedding vector
embedding_vector = np.random.rand(256).astype(np.float32).tolist()  # Generate random mass spectrum embedding
embedding_blob = serialize_f32(embedding_vector)
embedding = Embedding(compound_id=compound.id, vector=embedding_blob)
session.add(embedding)
session.commit()

# Close the session
session.close()

# ========================== Similarity search using sqlite-vec ==========================

db = sqlite3.connect("chemistry2.db")
db.enable_load_extension(True)
sqlite_vec.load(db)
db.enable_load_extension(False)

# Create sqlite-vec virtual tables
db.execute("CREATE VIRTUAL TABLE IF NOT EXISTS embedding_vec USING vec0(embedding float[256])")

query_vector = np.random.rand(256).astype(np.float32).tolist()

rows = db.execute(
    """
      SELECT
        rowid,
        distance
      FROM embedding_vec
      WHERE embedding MATCH ?
        AND k = 1
      ORDER BY distance
    """,
    [serialize_f32(query_vector)],
).fetchall()

print(rows)

db.close()


2025-03-20 17:50:10,446 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-03-20 17:50:10,447 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("compound")
2025-03-20 17:50:10,448 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-03-20 17:50:10,450 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("compound")
2025-03-20 17:50:10,451 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-03-20 17:50:10,452 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("property")
2025-03-20 17:50:10,453 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-03-20 17:50:10,454 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("property")
2025-03-20 17:50:10,455 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-03-20 17:50:10,456 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("spectrum")
2025-03-20 17:50:10,456 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-03-20 17:50:10,458 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("spectrum")
2025-03-20 17:50:10,458 INFO sqlalchemy.engine.Engine [raw s

# 6.4 Approaches to Acquiring Database Information  

## 6.4.1 Programmatic access using Python requests library

In [3]:
import requests
import pubchempy as pc
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import json

def search_pubchem(formula, timeout=999):
    # get pubchem cid based on formula
    cids = pc.get_cids(formula, 'formula', list_return='flat')
    idstring = ''
    smiles = []
    inchikey = []
    all_cids = []
    # search pubchem via formula with pug
    for i, cid in enumerate(cids):
        idstring += ',' + str(cid)
        if ((i%100==99) or (i==len(cids)-1)):
            url_i = "http://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/" + idstring[1:(len(idstring))] + "/property/InChIKey,CanonicalSMILES/JSON"
            res_i = requests.get(url_i, timeout=timeout)
            soup_i = BeautifulSoup(res_i.content, "html.parser")
            str_i = str(soup_i)
            properties_i = json.loads(str_i)['PropertyTable']['Properties']
            idstring = ''
            for properties_ij in properties_i:
                smiles_ij = properties_ij['CanonicalSMILES']
                if smiles_ij not in smiles:
                    smiles.append(smiles_ij)
                    inchikey.append(properties_ij['InChIKey'])
                    all_cids.append(str(properties_ij['CID']))
                else:
                    wh = np.where(np.array(smiles)==smiles_ij)[0][0]
                    all_cids[wh] = all_cids[wh] + ', ' + str(properties_ij['CID'])
    
    result = pd.DataFrame({'InChIKey': inchikey, 'SMILES': smiles, 'PubChem': all_cids})
    return result

print(search_pubchem('C3H6O')[:10])

                      InChIKey   SMILES  \
0  CSCPPACGZOOCGX-UHFFFAOYSA-N  CC(=O)C   
1  XXROGKLTLUQVRX-UHFFFAOYSA-N    C=CCO   
2  NBBJYMSMWIIQGU-UHFFFAOYSA-N    CCC=O   
3  GOOHAUXETOMSMM-UHFFFAOYSA-N   CC1CO1   
4  XJRBAMWJDBPFIM-UHFFFAOYSA-N    COC=C   
5  AHHWIHXENZJRFG-UHFFFAOYSA-N   C1COC1   
6  YOXHCYXIAVIFCZ-UHFFFAOYSA-N   C1CC1O   
7  NARVIWMVBMUEOG-UHFFFAOYSA-N  CC(=C)O   
8  DOKHEARVIDLSFF-NSCUHMNNSA-N    CC=CO   
9  AAXILUCBQRGWQB-UHFFFAOYSA-N   CO.C#C   

                                             PubChem  
0  180, 522220, 6912034, 11007810, 13001281, 1143...  
1  7858, 12239028, 16212955, 21365460, 71309116, ...  
2  527, 56845944, 12201346, 71309167, 12201347, 1...  
3  6378, 146261, 146262, 16212941, 12216770, 1221...  
4  7861, 12322432, 12322431, 59134988, 12322433, ...  
5    10423, 90745093, 23270599, 142791746, 157115188  
6                                  123361, 172405830  
7             141483, 101632558, 59274059, 159443210  
8  5463330, 5463331, 143500, 23

## 6.4.2 Selenium Automation

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By

def get_properties_with_selenium(cid):
    driver = webdriver.Edge()
    try:
        driver.get(f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}")
        
        # Wait for page load
        driver.implicitly_wait(10)
        
        # Get properties data
        properties_section = driver.find_element(By.ID, "Chemical-and-Physical-Properties")
        print(properties_section.text)
        
    finally:
        driver.quit()

get_properties_with_selenium(2244)

3 Chemical and Physical Properties
3.1 Computed Properties
Property Name Property Value Reference
Molecular Weight
180.16 g/mol
Computed by PubChem 2.2 (PubChem release 2021.10.14)
XLogP3
1.2
Computed by XLogP3 3.0 (PubChem release 2021.10.14)
Hydrogen Bond Donor Count
1
Computed by Cactvs 3.4.8.18 (PubChem release 2021.10.14)
Hydrogen Bond Acceptor Count
4
Computed by Cactvs 3.4.8.18 (PubChem release 2021.10.14)
Rotatable Bond Count
3
Computed by Cactvs 3.4.8.18 (PubChem release 2021.10.14)
Exact Mass
180.04225873 Da
Computed by PubChem 2.2 (PubChem release 2021.10.14)
Monoisotopic Mass
180.04225873 Da
Computed by PubChem 2.2 (PubChem release 2021.10.14)
Topological Polar Surface Area
63.6 Å²
Computed by Cactvs 3.4.8.18 (PubChem release 2021.10.14)
Heavy Atom Count
13
Computed by PubChem
Formal Charge
0
Computed by PubChem
Complexity
212
Computed by Cactvs 3.4.8.18 (PubChem release 2021.10.14)
Isotope Atom Count
0
Computed by PubChem
Defined Atom Stereocenter Count
0
Computed by PubCh

## 6.4.2 Using AI-powered tools like Browser use

In [ ]:
from langchain_openai import ChatOpenAI
from browser_use import Agent
from dotenv import load_dotenv
import nest_asyncio
import asyncio

load_dotenv()
nest_asyncio.apply()

llm = ChatOpenAI(model="gpt-4o")

async def main():
    agent = Agent(
        task="Open PubChem, search for aspirin, and download data used to display the page of the first search results in json format",
        llm=llm,
    )
    result = await agent.run()
    print(result)
    
if __name__ == '__main__':
    asyncio.run(main())